In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib  # For saving model (optional)


In [2]:
# Load from attached file (matches scikit-learn California Housing)
df = pd.read_csv('D:/MLE/Assignment_3/housing.csv')

# Feature and target info (standard for this dataset)
feature_names = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 
                 'total_bedrooms', 'population', 'households', 'median_income']
target_name = 'median_house_value'

X = df[feature_names]
y = df[target_name]

print("Feature names:", feature_names)
print("Target name:", target_name)
print("Number of samples:", len(df))
print("Number of features:", len(feature_names))
print("Dataset shape:", df.shape)


Feature names: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
Target name: median_house_value
Number of samples: 20640
Number of features: 8
Dataset shape: (20640, 10)


In [3]:
print("First 10 rows:")
print(df.head(10))


First 10 rows:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -122.23     37.88                41.0        880.0           129.0   
1    -122.22     37.86                21.0       7099.0          1106.0   
2    -122.24     37.85                52.0       1467.0           190.0   
3    -122.25     37.85                52.0       1274.0           235.0   
4    -122.25     37.85                52.0       1627.0           280.0   
5    -122.25     37.85                52.0        919.0           213.0   
6    -122.25     37.84                52.0       2535.0           489.0   
7    -122.25     37.84                52.0       3104.0           687.0   
8    -122.26     37.84                42.0       2555.0           665.0   
9    -122.25     37.84                52.0       3549.0           707.0   

   population  households  median_income  median_house_value ocean_proximity  
0       322.0       126.0         8.3252            452600.0        NEAR BAY  
1

In [4]:
print("Missing values per column:")
print(df.isnull().sum())


Missing values per column:
longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


In [5]:
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_names)
print("Shape after imputation:", X_imputed.shape)
print("Missing values after:", X_imputed.isnull().sum().sum())


Shape after imputation: (20640, 8)
Missing values after: 0


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42
)
print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


Training samples: 16512
Test samples: 4128


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data mean after scaling (should ~0):", np.mean(X_train_scaled, axis=0).round(2))
print("Training data std after scaling (should ~1):", np.std(X_train_scaled, axis=0).round(2))


Training data mean after scaling (should ~0): [ 0.  0. -0.  0. -0. -0. -0. -0.]
Training data std after scaling (should ~1): [1. 1. 1. 1. 1. 1. 1. 1.]


In [8]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)
print("Model coefficients shape:", model.coef_.shape)
print("Model trained successfully!")


Model coefficients shape: (8,)
Model trained successfully!


In [9]:
y_pred = model.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.4f}")

print("\nInterpretation:")
print("- RMSE 71133 means average error of ~$71k (target in $ units).")
print("- R² 0.6139: Model explains 61% of house price variance; solid linear baseline.")
print("- MSE 5.06e9 shows scale of squared errors; typical for $200k+ prices.")
print("- Improve with feature engineering (rooms/household) or RandomForestRegressor.")



MSE: 5059928371.17
RMSE: 71133.17
R² Score: 0.6139

Interpretation:
- RMSE 71133 means average error of ~$71k (target in $ units).
- R² 0.6139: Model explains 61% of house price variance; solid linear baseline.
- MSE 5.06e9 shows scale of squared errors; typical for $200k+ prices.
- Improve with feature engineering (rooms/household) or RandomForestRegressor.


In [10]:
predictions_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_pred
})
predictions_df.to_csv('predictions.csv', index=False)
print("Predictions saved to predictions.csv (first 5 rows):")
print(predictions_df.head())


Predictions saved to predictions.csv (first 5 rows):
     actual      predicted
0   47700.0   63736.591338
1   45800.0  154344.594319
2  500001.0  253073.194287
3  218600.0  263507.746536
4  278000.0  266883.359611
